# U07_P — Phylogenetic Trees — Solutions
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 7:** Phylogenetic Analysis
**Type:** Practice Solutions notebook · **Time:** 50 min · **Difficulty:** ★★☆
**Goes with:** Lectures L18–L20, Practical S8
**Before this:** U07_C1

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U07_P"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

Python 3.11.18 | Biopython 1.88


### Ex 7.1 ★☆☆ — Calculate pairwise distance matrix for globin family
**Biology:** Pairwise sequence divergence quantifies evolutionary time since common ancestry.  
**Task:** Compute the identity distance matrix for `globin_family.aln` using `DistanceCalculator('identity')`. Store in variable `dist_mat`.  
**Expected:** `dist_mat` as a Bio.Phylo `DistanceMatrix` object with 7 taxa.

In [2]:
from Bio import AlignIO
from Bio.Phylo.TreeConstruction import DistanceCalculator

aln = AlignIO.read(DATA_DIR / "alignments" / "globin_family.aln", "clustal")
calc = DistanceCalculator("identity")
dist_mat = calc.get_distance(aln)

In [3]:
# Check — do not edit
assert len(dist_mat.names) == 7, f"Expected 7 taxa, got {len(dist_mat.names)}"
print("Ex 7.1 passed")

Ex 7.1 passed


### Ex 7.2 ★☆☆ — Identify the minimum evolutionary distance pair
**Biology:** The two most similar paralogues in the globin family represent the most recent gene duplication.  
**Task:** Find the pair of distinct taxa with the smallest distance in `dist_mat`. Store their names in a tuple `closest_pair` and their distance in `min_dist`.  
**Expected:** `closest_pair` containing 'HBB_HUMAN' and 'HBD_HUMAN', `min_dist` < 0.10.

In [4]:
names = dist_mat.names
min_dist = 999.0
closest_pair = ()

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        d = dist_mat[i, j]
        if d < min_dist:
            min_dist = d
            closest_pair = (names[i], names[j])

In [5]:
# Check — do not edit
assert "HBB_HUMAN" in closest_pair and "HBD_HUMAN" in closest_pair
assert min_dist < 0.10, f"Expected min_dist < 0.10, got {min_dist}"
print("Ex 7.2 passed")

Ex 7.2 passed


### Ex 7.3 ★☆☆ — Construct a Neighbor Joining tree
**Biology:** Neighbor Joining is a computationally efficient distance method that does not assume a constant molecular clock.  
**Task:** Construct a Neighbor Joining tree from `dist_mat` using `DistanceTreeConstructor().nj()`. Store in `nj_tree`. Verify it has 7 terminal taxa.  
**Expected:** `nj_tree` with 7 terminal clades.

In [6]:
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor

constructor = DistanceTreeConstructor()
nj_tree = constructor.nj(dist_mat)
num_taxa = len(nj_tree.get_terminals())

In [7]:
# Check — do not edit
assert num_taxa == 7, f"Expected 7 terminal taxa, got {num_taxa}"
print("Ex 7.3 passed")

Ex 7.3 passed


### Ex 7.4 ★★☆ — Construct UPGMA tree for vertebrate insulin family
**Biology:** Insulin is an essential metabolic hormone conserved across all vertebrates.  
**Task:** Load `data/alignments/insulin_family.aln`. Compute the distance matrix with `DistanceCalculator('identity')`, and construct a UPGMA tree in `ins_upgma_tree`.  
**Expected:** `ins_upgma_tree` with terminal taxa matching insulin alignment.

In [8]:
ins_aln = AlignIO.read(DATA_DIR / "alignments" / "insulin_family.aln", "clustal")
ins_dm = calc.get_distance(ins_aln)
ins_upgma_tree = constructor.upgma(ins_dm)

In [9]:
# Check — do not edit
assert len(ins_upgma_tree.get_terminals()) == len(ins_aln)
print("Ex 7.4 passed")

Ex 7.4 passed


### Ex 7.5 ★★☆ — Identify closest primate sister pair in the insulin tree
**Biology:** Human and chimpanzee insulins share nearly 100% sequence identity.  
**Task:** Extract the pairwise distance between `'INS_HUMAN'` and `'INS_CHIMP'` from `ins_dm` into `human_chimp_dist`.  
**Expected:** `human_chimp_dist` as float == 0.0.

In [10]:
human_chimp_dist = ins_dm["INS_HUMAN", "INS_CHIMP"]

In [11]:
# Check — do not edit
assert human_chimp_dist == 0.0, f"Expected 0.0 distance between human and chimp insulin, got {human_chimp_dist}"
print("Ex 7.5 passed")

Ex 7.5 passed


### Ex 7.6 ★★☆ — Classify gene relationships as Orthologues or Paralogues
**Biology:** Speciation vs duplication determines functional homology.  
**Task:** Given two gene comparisons: (1) Human HBB vs Chimpanzee HBB (`comp1`), and (2) Human HBB vs Human HBD (`comp2`), assign `'Orthologues'` or `'Paralogues'` to `rel1` and `rel2`.  
**Expected:** `rel1` = 'Orthologues', `rel2` = 'Paralogues'.

In [12]:
rel1 = "Orthologues"
rel2 = "Paralogues"

In [13]:
# Check — do not edit
assert rel1 == "Orthologues", "Genes separated by speciation are orthologues"
assert rel2 == "Paralogues", "Genes separated by gene duplication within the same genome are paralogues"
print("Ex 7.6 passed")

Ex 7.6 passed


### Ex 7.7 ★★★ — Compare total branch length between UPGMA and NJ trees
**Biology:** Neighbor Joining allows varying evolutionary branch lengths across fast- and slow-evolving lineages.  
**Task:** Calculate the total tree branch length (sum of all branch lengths) for `nj_tree` using `nj_tree.total_branch_length()` into `nj_length`, and `upgma_tree.total_branch_length()` into `upgma_length`.  
**Expected:** Both `nj_length` and `upgma_length` are positive floats.

In [14]:
upgma_tree = constructor.upgma(dist_mat)
upgma_length = upgma_tree.total_branch_length()
nj_length = nj_tree.total_branch_length()

In [15]:
# Check — do not edit
assert isinstance(upgma_length, float) and upgma_length > 0
assert isinstance(nj_length, float) and nj_length > 0
print("Ex 7.7 passed")

Ex 7.7 passed


## 5. Reflection Questions
1. Why does the UPGMA clustering algorithm produce an ultrametric tree, and why does this fail when different organismal lineages evolve at different rates?
2. What is the biological consequence of assuming two genes are orthologues when they are actually ancient paralogues?